# Behavioural Re-identification of Renamed LLM Agents
### A Leakage-Controlled Evaluation on Real Multi-Agent Logs

Companion notebook to the report by Mewni Alahakoon (Swarm Alignment Hackathon, Apart Research, November 2026). It regenerates the results in the report from the raw AI Village data on a free Colab CPU runtime. See the repository README for the mapping from each table and figure to the experiment that produces it.

**Question.** If AI agents rename themselves, can an investigator still tell which names belong to the same agent, using only how they write and when they post?

**Method in one line.** Take AI Village chat, where the author of every message is known, mask all agent and model names, split each agent's time-ordered messages into aliases of 20 messages, and test whether unsupervised similarity links the aliases back together. No model is trained; identities are used only to score.

**How to run**
1. Request access to the AI Village dataset on Hugging Face (`aidigestorg/ai-village`) and create a read token.
2. In Colab, add the token under **Secrets** (key icon on the left) as `HF_TOKEN` and allow notebook access. Never paste a token into a cell.
3. **Runtime → Run all.** A CPU runtime is enough. Results (CSV and PNG) are saved to `MyDrive/swarm-alignment/results`.
4. No data access yet? Set `USE_SYNTHETIC_DEMO = True` in the CONFIG cell to test the pipeline on synthetic data. Demo numbers are not results.

**Data terms.** Analysis only: nothing here trains or fine-tunes a model, only AI agents are analysed (human messages are removed), and no data is redistributed. Cite AI Digest (2026), *AI Village dataset*.

In [ ]:
# ---- Setup ----
!pip -q install huggingface_hub
import os, sys, json, time, warnings
warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata, drive
    drive.mount("/content/drive")
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:          # no secret yet: fine for the synthetic demo, required for the real data
        HF_TOKEN = None
    BASE = "/content/drive/MyDrive/swarm-alignment"
else:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    BASE = os.path.abspath("swarm-alignment")
DATA_DIR = f"{BASE}/data"
RESULTS_DIR = f"{BASE}/results"
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)
print("Saving to", BASE)

In [ ]:
# ---- CONFIG: the only cell you should need to edit ----
USE_SYNTHETIC_DEMO = False      # True = test the pipeline on fake data. NEVER report demo numbers.
REPO = "aidigestorg/ai-village"
VILLAGE = "main-village"        # also try "swarm-village"

# Leave as None to auto-detect; set to a column name if detection picks the wrong one
TEXT_COL = None
AUTHOR_COL = None
TIME_COL = None
AGENT_ALLOWLIST = None          # e.g. ["Claude Opus 4.1", "GPT-5", ...] if human filtering fails

MIN_MSGS = 80                   # agents need at least this many messages to be included
MIN_CHARS = 20                  # ignore very short messages ("ok", "thanks")
MAX_ROWS = None                 # set e.g. 300_000 if Colab runs out of RAM
MSGS_PER_ALIAS = 20             # main setting: each fake alias has this many messages
ALIAS_SIZES = [5, 10, 20, 40]   # for the "how much data do you need" curve
MAX_ALIASES_PER_AGENT = 30      # balance agents so prolific ones don't dominate
SEEDS = [0, 1, 2]               # repeat with different random splits, report mean ± std

BLOCKS_ALL = ["char n-grams", "word n-grams", "function words", "style stats", "timing"]
BLOCKS_STYLE_ONLY = ["function words", "style stats"]   # topic-light features

In [ ]:
# ---- Look at what's in the village folder (no big downloads) ----
from huggingface_hub import HfApi, hf_hub_download
if not USE_SYNTHETIC_DEMO:
    api = HfApi(token=HF_TOKEN)
    DATASET_SHA = api.dataset_info(REPO, token=HF_TOKEN).sha      # the data is refreshed weekly: record the snapshot
    open(f"{RESULTS_DIR}/dataset_version.txt", "w").write(f"{REPO} commit {DATASET_SHA}, downloaded {time.strftime('%Y-%m-%d')}\n")
    print(f"Dataset snapshot: {REPO} @ {DATASET_SHA[:7]} (cite this in the report)")
    files = api.list_repo_files(REPO, repo_type="dataset", token=HF_TOKEN)
    village_files = [f for f in files if f.startswith(VILLAGE + "/") and "/images/" not in f and not f.endswith((".png", ".jpg", ".webp"))]
    print(f"{len(village_files)} non-image files in {VILLAGE}:")
    for f in village_files[:60]:
        print("  ", f)
    for f in ["README.md", "SCHEMA.md"]:
        hf_hub_download(REPO, f, repo_type="dataset", token=HF_TOKEN, local_dir=DATA_DIR)
    schema = open(f"{DATA_DIR}/SCHEMA.md", encoding="utf-8").read()
    # print the parts of the schema about chat messages and agents
    for sec in schema.split("\n## "):
        if any(k in sec.lower()[:200] for k in ["chat", "agent", "message"]):
            print("\n## " + sec[:2500])

In [ ]:
# ---- Download only the chat messages + agent list for this village ----
def pick(pattern_list):
    for p in pattern_list:
        hits = [f for f in village_files if os.path.basename(f).startswith(p)]
        if hits:
            return sorted(hits, key=len)[0]
    return None

if not USE_SYNTHETIC_DEMO:
    CHAT_FILE = pick(["chat_messages", "messages", "chat"])
    AGENTS_FILE = pick(["agents"])
    assert CHAT_FILE, "No chat file found in this village. Check the file list above and set CHAT_FILE manually."
    for f in [CHAT_FILE, AGENTS_FILE]:
        if f:
            t = time.time()
            hf_hub_download(REPO, f, repo_type="dataset", token=HF_TOKEN, local_dir=DATA_DIR)
            print(f"downloaded {f} in {time.time()-t:.0f}s")

In [ ]:
# ==== CORE: loading, cleaning, alias simulation, features, evaluation ====
import gzip, json, re, random, math, os
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.metrics import roc_auc_score, adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.cluster import AgglomerativeClustering


# ---------- loading ----------
def read_jsonl(path, limit=None):
    opener = gzip.open if path.endswith(".gz") else open
    rows = []
    with opener(path, "rt", encoding="utf-8") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
            if limit and len(rows) >= limit:
                break
    return pd.json_normalize(rows, max_level=1)


def find_col(df, candidates, override=None, required=True, label=""):
    """Find a column by exact name, then by suffix (for flattened 'author.id'), then by substring."""
    if override:
        if override not in df.columns:
            raise KeyError(f"{label}: column '{override}' not found. Columns: {list(df.columns)}")
        return override
    cols = list(df.columns)
    low = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand in low:
            return low[cand]
    for cand in candidates:
        for c in cols:
            if c.lower().endswith("." + cand):
                return c
    for cand in candidates:
        for c in cols:
            if cand in c.lower():
                return c
    if required:
        raise KeyError(f"Could not auto-detect the {label} column. Set it in the config cell. Columns: {cols}")
    return None


def to_text(v):
    """Messages may be plain strings, dicts, or lists of content parts."""
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return ""
    if isinstance(v, str):
        return v
    if isinstance(v, dict):
        for k in ("text", "content", "body", "message"):
            if k in v:
                return to_text(v[k])
        return ""
    if isinstance(v, (list, tuple)):
        return "\n".join(t for t in (to_text(x) for x in v) if t)
    return str(v)


def to_time(s):
    if s is None:
        return None
    if pd.api.types.is_numeric_dtype(s):
        unit = "ms" if s.dropna().median() > 1e11 else "s"
        return pd.to_datetime(s, unit=unit, errors="coerce", utc=True)
    try:
        return pd.to_datetime(s, errors="coerce", utc=True, format="mixed")
    except (TypeError, ValueError):
        return pd.to_datetime(s, errors="coerce", utc=True)


# ---------- model family (for the "copies of the same model" analysis) ----------
FAMILY_KEYS = [("claude", "claude"), ("opus", "claude"), ("sonnet", "claude"), ("haiku", "claude"),
               ("fable", "claude"), ("mythos", "claude"),
               ("gpt", "openai"), ("o1", "openai"), ("o3", "openai"), ("o4", "openai"), ("openai", "openai"),
               ("gemini", "google"), ("gemma", "google"),
               ("grok", "xai"), ("deepseek", "deepseek"), ("llama", "meta"), ("qwen", "qwen"),
               ("mistral", "mistral"), ("kimi", "moonshot")]


def model_family(s):
    s = str(s).lower()
    for key, fam in FAMILY_KEYS:
        if key in s:
            return fam
    return "unknown"


# ---------- name masking (stops the trivial "agent signs its own name" leak) ----------
MODEL_WORDS = ["claude", "opus", "sonnet", "haiku", "gpt", "chatgpt", "openai", "anthropic", "gemini",
               "google", "deepmind", "grok", "xai", "deepseek", "llama", "qwen", "mistral", "o1", "o3", "o4",
               "fable", "mythos", "kimi"]
COMMON = set("the and for with from this that agent bot model village team mini pro flash max".split())


def build_mask_regex(names):
    toks = set(MODEL_WORDS)
    for n in names:
        n = str(n).strip().lower()
        if not n:
            continue
        toks.add(n)
        for t in re.split(r"[\s\-_\.]+", n):
            if len(t) >= 3 and t not in COMMON and not t.isdigit():
                toks.add(t)
    toks = sorted(toks, key=len, reverse=True)
    return re.compile(r"(?i)(?<![A-Za-z0-9])@?(" + "|".join(re.escape(t) for t in toks) + r")(?![A-Za-z0-9])")


# ---------- alias simulation ----------
def make_aliases(msgs, m, mode="contiguous", max_per_agent=30, seed=0):
    """Split each agent's messages into pseudo-aliases of m messages each.
    contiguous = consecutive messages in time (closest to an agent renaming itself over time)
    random     = messages shuffled before chunking (an easier, topic-mixed baseline)"""
    rng = random.Random(seed)
    out = []
    for agent, g in msgs.groupby("agent"):
        g = g.sort_values("time") if "time" in g and g["time"].notna().any() else g
        idx = list(range(len(g)))
        if mode == "random":
            rng.shuffle(idx)
        start = rng.randrange(m) if mode == "contiguous" else 0   # seed changes where the cuts fall
        chunks = [idx[i:i + m] for i in range(start, len(idx) - m + 1, m)]
        if len(chunks) < 2:
            continue
        if len(chunks) > max_per_agent:
            chunks = sorted(rng.sample(chunks, max_per_agent))
        texts = g["text"].tolist()
        times = g["time"].tolist() if "time" in g else [None] * len(g)
        for k, ch in enumerate(chunks):
            out.append({"alias": f"{agent}#{k}", "agent": agent,
                        "texts": [texts[i] for i in ch], "times": [times[i] for i in ch]})
    al = pd.DataFrame(out)
    if len(al) and al["agent"].nunique() < 2:
        raise ValueError("Need at least 2 agents with enough messages. Lower MIN_MSGS or MSGS_PER_ALIAS.")
    return al


# ---------- features ----------
FUNCTION_WORDS = """a about above after again against all am an and any are as at be because been before being
below between both but by can could did do does doing down during each few for from further had has have having he
her here hers herself him himself his how i if in into is it its itself just me more most my myself no nor not of off
on once only or other our ours ourselves out over own same she should so some such than that the their theirs them
themselves then there these they this those through to too under until up very was we were what when where which while
who whom why will with would you your yours yourself yourselves also however perhaps maybe really actually basically
let lets great sure okay ok thanks thank please indeed quite rather still yet already even""".split()
FW_INDEX = {w: i for i, w in enumerate(dict.fromkeys(FUNCTION_WORDS))}

EMOJI_RE = re.compile("[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F000-\U0001F2FF]")
URL_RE = re.compile(r"https?://\S+")


def style_stats(texts):
    """Topic-light stylometric statistics aggregated over an alias's messages."""
    feats = []
    for t in texts:
        n = max(len(t), 1)
        words = re.findall(r"[A-Za-z']+", t)
        nw = max(len(words), 1)
        lines = t.split("\n")
        feats.append([
            math.log1p(len(t)), math.log1p(len(words)),
            np.mean([len(w) for w in words]) if words else 0,
            t.count("!") / n * 100, t.count("?") / n * 100, t.count(",") / n * 100,
            t.count(";") / n * 100, t.count(":") / n * 100, (t.count("—") + t.count("–")) / n * 100,
            t.count("...") / n * 100, t.count("*") / n * 100, t.count("#") / n * 100, t.count("`") / n * 100,
            t.count("\n") / n * 100, len(EMOJI_RE.findall(t)) / n * 100,
            sum(c.isupper() for c in t) / n, sum(c.isdigit() for c in t) / n,
            sum(1 for l in lines if re.match(r"\s*([-*•]|\d+\.)\s", l)) / max(len(lines), 1),
            len(URL_RE.findall(t)) / nw, t.count("(") / n * 100,
            sum(1 for w in words if w.lower() in ("i", "i'm", "i'll", "i've", "me", "my")) / nw,
            sum(1 for w in words if w.lower() in ("we", "we're", "we'll", "us", "our", "let's")) / nw,
            1.0 if t[:1].isupper() else 0.0,
        ])
    a = np.array(feats, dtype=float)
    return np.concatenate([a.mean(0), a.std(0)])


def function_word_vec(texts):
    v = np.zeros(len(FW_INDEX))
    total = 0
    for t in texts:
        for w in re.findall(r"[a-z']+", t.lower()):
            total += 1
            j = FW_INDEX.get(w)
            if j is not None:
                v[j] += 1
    return v / max(total, 1)


def timing_vec(times):
    ts = to_time(pd.Series(times)).dropna().sort_values()
    if len(ts) < 2:
        return None
    hours = np.bincount(ts.dt.hour, minlength=24) / len(ts)
    gaps = ts.diff().dt.total_seconds().dropna().clip(lower=1)
    lg = np.log(gaps)
    return np.concatenate([hours, [lg.median(), lg.std() if len(lg) > 1 else 0]])


def cosine_sim(X):
    X = normalize(X)
    S = X @ X.T
    return np.asarray(S.todense() if hasattr(S, "todense") else S)


def feature_blocks(aliases, use_timing=True):
    """Return {block name: alias-by-alias cosine similarity}. Every block is unsupervised:
    no labels are used and no model is trained (respects the AI Village terms)."""
    docs = ["\n".join(t) for t in aliases["texts"]]
    blocks = {}
    char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True, min_df=2, max_features=60000)
    blocks["char n-grams"] = cosine_sim(char.fit_transform(docs))
    word = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), sublinear_tf=True, min_df=2, max_features=60000,
                           token_pattern=r"(?u)\b\w+\b")
    blocks["word n-grams"] = cosine_sim(word.fit_transform(docs))
    fw = np.vstack([function_word_vec(t) for t in aliases["texts"]])
    blocks["function words"] = cosine_sim(StandardScaler().fit_transform(fw))
    st = np.vstack([style_stats(t) for t in aliases["texts"]])
    blocks["style stats"] = cosine_sim(StandardScaler().fit_transform(np.nan_to_num(st)))
    if use_timing:
        tv = [timing_vec(t) for t in aliases["times"]]
        if all(v is not None for v in tv):
            blocks["timing"] = cosine_sim(StandardScaler().fit_transform(np.vstack(tv)))
    return blocks


def combine(blocks, names):
    return np.mean([blocks[n] for n in names if n in blocks], axis=0)


# ---------- evaluation ----------
def evaluate(S, labels, families=None, estimate_k=True):
    labels = np.asarray(labels)
    n = len(labels)
    S = S.copy()
    np.fill_diagonal(S, -np.inf)
    order = np.argsort(-S, axis=1)
    same = labels[order] == labels[:, None]
    top1 = same[:, 0].mean()
    top5 = same[:, :5].any(1).mean()
    counts = Counter(labels)
    chance = np.mean([(counts[l] - 1) / (n - 1) for l in labels])
    iu = np.triu_indices(n, 1)
    y = (labels[:, None] == labels[None, :])[iu]
    Sf = S.copy(); np.fill_diagonal(Sf, 1.0)
    auc = roc_auc_score(y, Sf[iu])
    D = np.clip(1 - Sf, 0, None); np.fill_diagonal(D, 0)
    k_true = len(counts)
    cl = AgglomerativeClustering(n_clusters=k_true, metric="precomputed", linkage="average").fit_predict(D)
    res = {"aliases": n, "agents": k_true, "top1": top1, "top5": top5, "chance_top1": chance,
           "pair_auc": auc, "ARI": adjusted_rand_score(labels, cl), "NMI": normalized_mutual_info_score(labels, cl)}
    if estimate_k:
        best_k, best_s = None, -1
        for k in range(2, min(n - 1, 3 * k_true) + 1):
            lab = AgglomerativeClustering(n_clusters=k, metric="precomputed", linkage="average").fit_predict(D)
            if len(set(lab)) < 2:
                continue
            s = silhouette_score(D, lab, metric="precomputed")
            if s > best_s:
                best_k, best_s = k, s
        res["estimated_agents"] = best_k
    if families is not None:
        res.update(same_family_eval(S, labels, np.asarray(families)))
    return res


def same_family_eval(S, labels, fams):
    """Hard case: only compare against aliases from agents built on the SAME model family
    (closest analogue to the incident, where every agent was the same model)."""
    hits, base = [], []
    for i in range(len(labels)):
        pool = np.where((fams == fams[i]) & (np.arange(len(labels)) != i))[0]
        if len(set(labels[pool])) < 2:
            continue
        j = pool[np.argmax(S[i, pool])]
        hits.append(labels[j] == labels[i])
        base.append(np.mean(labels[pool] == labels[i]))
    if not hits:
        return {"same_family_top1": np.nan, "same_family_chance": np.nan, "same_family_n": 0}
    return {"same_family_top1": float(np.mean(hits)), "same_family_chance": float(np.mean(base)),
            "same_family_n": len(hits)}


def wrong_neighbour_same_family(S, labels, fams):
    """Of the top-1 mistakes, how many point at a different agent on the same model family?"""
    labels, fams = np.asarray(labels), np.asarray(fams)
    S = S.copy(); np.fill_diagonal(S, -np.inf)
    nn = S.argmax(1)
    wrong = labels[nn] != labels
    if not wrong.any():
        return np.nan, np.nan
    obs = np.mean(fams[nn][wrong] == fams[wrong])
    exp = np.mean([np.mean(fams[(labels != labels[i])] == fams[i]) for i in np.where(wrong)[0]])
    return obs, exp


# ---------- disguise transforms (rule-based "agent trying to hide its style") ----------
SWAPS = {"i'll": "i will", "i will": "i'll", "can't": "cannot", "cannot": "can't", "let's": "let us",
         "great": "good", "really": "very", "awesome": "nice", "definitely": "certainly",
         "okay": "ok", "ok": "okay", "thanks": "thank you", "however": "but", "additionally": "also",
         "i'm": "i am", "i am": "i'm", "it's": "it is", "don't": "do not", "won't": "will not"}
FILLERS = re.compile(r"(?i)\b(just|really|actually|basically|definitely|honestly|literally|quite|very)\s+")
SWAP_RE = re.compile(r"(?i)\b(" + "|".join(re.escape(k) for k in sorted(SWAPS, key=len, reverse=True)) + r")\b")


def d_strip_format(t):
    t = EMOJI_RE.sub("", t)
    t = re.sub(r"[*_`#>]+", "", t)
    t = re.sub(r"(?m)^\s*([-•]|\d+\.)\s+", "", t)
    t = re.sub(r"([!?.]){2,}", r"\1", t)
    t = t.replace("—", ", ").replace("–", ", ")
    t = re.sub(r"\s+", " ", t)
    return t.lower().strip()


def d_reword(t):
    t = FILLERS.sub("", t)
    return SWAP_RE.sub(lambda m: SWAPS.get(m.group(0).lower(), m.group(0)), t)


def d_truncate(t, max_words=25):
    first = re.split(r"(?<=[.!?])\s", t.strip(), maxsplit=1)[0]
    return " ".join(first.split()[:max_words])


DISGUISES = {
    "none": lambda t: t,
    "strip formatting": d_strip_format,
    "reword": d_reword,
    "shorten": d_truncate,
    "all combined": lambda t: d_truncate(d_reword(d_strip_format(t))),
}


def disguise_eval(aliases, block_names, disguise, seed=0, use_timing=True):
    """Half of each agent's aliases stay clean ('known'); the other half are disguised ('queries').
    Accuracy = share of disguised queries whose nearest known alias belongs to the right agent."""
    rng = np.random.default_rng(seed)
    al = aliases.copy().reset_index(drop=True)
    is_q = np.zeros(len(al), bool)
    for a, g in al.groupby("agent"):
        ix = g.index.to_numpy().copy(); rng.shuffle(ix)
        is_q[ix[: len(ix) // 2]] = True
    f = DISGUISES[disguise]
    al.loc[is_q, "texts"] = al.loc[is_q, "texts"].apply(lambda ts: [f(t) for t in ts])
    S = combine(feature_blocks(al, use_timing), block_names)
    q, k = np.where(is_q)[0], np.where(~is_q)[0]
    labels = al["agent"].to_numpy()
    nn = k[np.argmax(S[np.ix_(q, k)], axis=1)]
    acc = np.mean(labels[nn] == labels[q])
    counts = Counter(labels[k])
    chance = np.mean([counts[labels[i]] / len(k) for i in q])
    return acc, chance


# ---------- synthetic demo data (ONLY for testing the pipeline; never report these numbers) ----------
def make_synthetic_demo(n_agents=12, n_msgs=300, seed=0):
    rng = random.Random(seed)
    fams = ["Claude Opus", "Claude Sonnet", "Claude Haiku", "Claude Fable", "Claude Mythos",
            "GPT-5", "GPT-5 mini", "o3", "GPT-4.1", "Gemini 2.5 Pro", "Gemini Flash", "Gemini Ultra"]
    topics = ["the fundraiser", "the website", "the spreadsheet", "the blog post", "the event plan", "the email list"]
    openers = ["Great idea!", "I agree.", "Quick update:", "Hmm,", "Okay so", "Thanks!", "Noted —", "Honestly,"]
    agents, rows = [], []
    t0 = pd.Timestamp("2026-06-01", tz="UTC")
    for a in range(n_agents):
        name = fams[a]
        style = {"open": rng.sample(openers, 3), "bang": rng.random() < .5, "bullets": rng.random() < .4,
                 "emoji": rng.random() < .3, "long": rng.choice([8, 15, 30]), "we": rng.random() < .5,
                 "hour": rng.randrange(24)}
        sig = ["🦊", "🌿", "⚡", "🧭", "🎯", "🐙", "🌊", "🔭", "🧩", "🍀", "🛰️", "🪐"][a]
        agents.append({"id": f"agent_{a}", "name": name, "model": name, "emoji": sig})
        t = t0 + pd.Timedelta(minutes=rng.randrange(500))
        for i in range(n_msgs):
            topic = topics[(i // 40 + a) % len(topics)]
            words = " ".join(rng.choice(["really", "just", "maybe", "actually", "very", "also", "still"]) if rng.random() < .3
                             else rng.choice(["update", "work", "plan", "check", "draft", "share", "fix", "review"])
                             for _ in range(rng.randint(3, style["long"])))
            body = f"{rng.choice(style['open'])} {'We' if style['we'] else 'I'} {words} on {topic}"
            body += "!" if style["bang"] else "."
            if style["bullets"] and rng.random() < .5:
                body += "\n- " + words.split()[0] + "\n- next step"
            if style["emoji"] and rng.random() < .5:
                body += " 🚀"
            if a < 4 and rng.random() < .3:
                body += " " + sig                            # signature emoji the leak audit should flag
            if rng.random() < .1:
                body = f"This is {name} here. " + body      # name leak the masking should remove
            t = t + pd.Timedelta(minutes=rng.expovariate(1 / 90))
            rows.append({"id": len(rows), "author": {"id": f"agent_{a}", "name": name, "type": "agent"},
                         "content": body, "created_at": t.isoformat()})
    for i in range(400):  # human spectators, must be filtered out
        rows.append({"id": len(rows), "author": {"id": f"human_{i % 30}", "name": f"viewer{i % 30}", "type": "human"},
                     "content": "go agents go, this is so fun to watch", "created_at": (t0 + pd.Timedelta(hours=i)).isoformat()})
    rng.shuffle(rows)
    return pd.json_normalize(rows, max_level=1), pd.DataFrame(agents)

In [ ]:
# ---- Load + auto-detect columns ----
if USE_SYNTHETIC_DEMO:
    raw, agents_df = make_synthetic_demo()
else:
    raw = read_jsonl(f"{DATA_DIR}/{CHAT_FILE}", limit=MAX_ROWS)
    agents_df = read_jsonl(f"{DATA_DIR}/{AGENTS_FILE}") if AGENTS_FILE else pd.DataFrame()
print("chat rows:", len(raw)); print("chat columns:", list(raw.columns))
print("agents columns:", list(agents_df.columns))

text_col = find_col(raw, ["content", "text", "message", "body", "msg"], TEXT_COL, label="text")
author_col = find_col(raw, ["agent_id", "author_id", "sender_id", "agent_name", "author_name", "sender_name",
                            "author", "sender", "agent", "from", "speaker", "user", "name"], AUTHOR_COL, label="author")
time_col = find_col(raw, ["created_at", "timestamp", "sent_at", "time", "ts", "date"], TIME_COL, required=False, label="time")
print(f"\nUsing  text='{text_col}'  author='{author_col}'  time='{time_col}'")
display(raw[[c for c in [author_col, time_col, text_col] if c]].head(5))
print("\nTop authors:"); print(raw[author_col].astype(str).value_counts().head(25))

In [ ]:
# ---- Clean: keep AI agents only, mask names, drop tiny messages ----
msgs = pd.DataFrame({"agent": raw[author_col].astype(str),
                     "text": raw[text_col].apply(to_text),
                     "time": to_time(raw[time_col]) if time_col else pd.NaT})

# 1) keep agents only (removes human chat participants)
agent_ids, agent_names, agent_model = set(), set(), {}
if len(agents_df):
    id_c = find_col(agents_df, ["id", "agent_id", "slug"], required=False)
    nm_c = find_col(agents_df, ["name", "display_name", "agent_name"], required=False)
    md_c = find_col(agents_df, ["model", "model_name", "base_model", "llm"], required=False)
    for _, r in agents_df.iterrows():
        keys = [str(r[c]) for c in (id_c, nm_c) if c and pd.notna(r[c])]
        agent_ids.update(keys)
        if nm_c and pd.notna(r[nm_c]): agent_names.add(str(r[nm_c]))
        model = r[md_c] if md_c and pd.notna(r[md_c]) else (r[nm_c] if nm_c else "")
        for k in keys: agent_model[k] = str(model)
if AGENT_ALLOWLIST:
    keep = msgs["agent"].isin(AGENT_ALLOWLIST)
elif agent_ids and msgs["agent"].isin(agent_ids).any():
    keep = msgs["agent"].isin(agent_ids)
else:
    type_c = find_col(raw, ["author_type", "sender_type", "role", "is_agent", "type"], required=False)
    keep = raw[type_c].astype(str).str.lower().isin(["agent", "ai", "assistant", "true", "bot"]) if type_c else pd.Series(True, index=msgs.index)
    print("⚠️ Could not match authors to the agent list. Check 'Top authors' above; set AGENT_ALLOWLIST if humans slipped in.")
print(f"kept {keep.sum()} / {len(msgs)} messages as agent messages")
msgs = msgs[keep.values].copy()

# 2) mask every agent name and model name inside the text, so the method can't just read signatures
mask_re = build_mask_regex(list(agent_names) + list(msgs["agent"].unique()))
msgs["text"] = msgs["text"].str.replace(mask_re, "[AGENT]", regex=True)

# 3) drop short and exactly duplicated messages, keep agents with enough data
msgs = msgs[msgs["text"].str.strip().str.len() >= MIN_CHARS]
msgs = msgs.drop_duplicates(["agent", "text"])
counts = msgs["agent"].value_counts()
msgs = msgs[msgs["agent"].isin(counts[counts >= MIN_MSGS].index)].reset_index(drop=True)
msgs["family"] = msgs["agent"].map(lambda a: model_family(agent_model.get(a, a)))

summary = msgs.groupby("agent").agg(messages=("text", "size"), family=("family", "first"),
                                    first=("time", "min"), last=("time", "max")).sort_values("messages", ascending=False)
summary.to_csv(f"{RESULTS_DIR}/agents_included.csv")
print(f"\n{len(summary)} agents included, {len(msgs)} messages"); display(summary)
assert len(summary) >= 3, "Too few agents. Lower MIN_MSGS, or try another VILLAGE."

In [ ]:
# ---- Experiment 1: can we re-link aliases? Which features matter? (mean ± std over seeds) ----
ABLATIONS = {
    "all features": BLOCKS_ALL,
    "all text (no timing)": [b for b in BLOCKS_ALL if b != "timing"],
    "style only (function words + stats)": BLOCKS_STYLE_ONLY,
    "char n-grams": ["char n-grams"], "word n-grams": ["word n-grams"],
    "function words": ["function words"], "style stats": ["style stats"], "timing": ["timing"],
}
rows, CACHE = [], {}
for seed in SEEDS:
    al = make_aliases(msgs, MSGS_PER_ALIAS, "contiguous", MAX_ALIASES_PER_AGENT, seed)
    fams = al["agent"].map(msgs.groupby("agent")["family"].first()).to_numpy()
    blocks = feature_blocks(al)
    CACHE[seed] = (al, blocks, fams)
    for name, bl in ABLATIONS.items():
        if not any(b in blocks for b in bl):
            continue
        r = evaluate(combine(blocks, bl), al["agent"], fams, estimate_k=(name == "all features"))
        rows.append({"features": name, "seed": seed, **r})
e1 = pd.DataFrame(rows)
e1.to_csv(f"{RESULTS_DIR}/e1_ablation_raw.csv", index=False)
cols = ["top1", "top5", "pair_auc", "ARI", "same_family_top1"]
E1 = e1.groupby("features", sort=False)[cols + ["chance_top1", "same_family_chance"]].agg(["mean", "std"]).round(3)
E1.to_csv(f"{RESULTS_DIR}/e1_ablation_summary.csv")
print(f"aliases per run: {e1['aliases'].iloc[0]}, agents: {e1['agents'].iloc[0]}, {MSGS_PER_ALIAS} msgs/alias, contiguous")
display(E1)
est = e1[e1.features == "all features"]["estimated_agents"]
print(f"Blind estimate of how many agents are behind the aliases: {est.tolist()} (true: {e1['agents'].iloc[0]})")

In [ ]:
# ---- Shared chart style ----
import matplotlib.pyplot as plt
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]   # fixed categorical order
INK, MUTED, GRID = "#1f1f1e", "#6b6a64", "#e6e5e0"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": MUTED,
                     "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True,
                     "grid.color": GRID, "grid.linewidth": 0.8, "font.size": 11, "lines.linewidth": 2,
                     "axes.titleweight": "bold", "axes.titlesize": 12, "legend.frameon": False})

# Figure 1: ablation bars (top-1 accuracy) with chance line
m = e1.groupby("features", sort=False)["top1"].agg(["mean", "std"])
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.barh(m.index[::-1], m["mean"][::-1], xerr=m["std"][::-1], color=PAL[0], height=0.6, error_kw={"ecolor": MUTED, "lw": 1})
ch = e1["chance_top1"].mean()
ax.axvline(ch, color=MUTED, ls="--", lw=1.2); ax.text(ch + 0.01, len(m) - 0.45, f"chance {ch:.2f}", color=MUTED, fontsize=9, va="bottom")
for y, v in enumerate(m["mean"][::-1]): ax.text(v + 0.02, y, f"{v:.2f}", va="center", color=INK, fontsize=9)
ax.set_xlim(0, 1.08); ax.set_xlabel("Top-1 re-linking accuracy"); ax.grid(axis="y", visible=False)
ax.set_ylim(-0.6, len(m) - 0.1)
ax.set_title(f"Re-linking accuracy by feature set ({MSGS_PER_ALIAS} msgs/alias)", loc="left")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig1_ablation.png", dpi=200); plt.show()

In [ ]:
# ---- Experiment 2: how many messages does an alias need before it gives itself away? ----
rows = []
for m_size in ALIAS_SIZES:
    for seed in SEEDS:
        al = make_aliases(msgs, m_size, "contiguous", MAX_ALIASES_PER_AGENT, seed)
        blocks = feature_blocks(al)
        for name in ["all features", "style only (function words + stats)", "char n-grams"]:
            r = evaluate(combine(blocks, ABLATIONS[name]), al["agent"], estimate_k=False)
            rows.append({"msgs_per_alias": m_size, "features": name, "seed": seed, **r})
e2 = pd.DataFrame(rows); e2.to_csv(f"{RESULTS_DIR}/e2_data_curve.csv", index=False)
E2 = e2.groupby(["features", "msgs_per_alias"])[["top1", "pair_auc", "chance_top1"]].mean().round(3)
display(E2)

fig, ax = plt.subplots(figsize=(9, 3.8))
for i, name in enumerate(["all features", "style only (function words + stats)", "char n-grams"]):
    g = e2[e2.features == name].groupby("msgs_per_alias")["top1"].agg(["mean", "std"])
    ax.errorbar(g.index, g["mean"], yerr=g["std"], marker="o", ms=8, color=PAL[i], capsize=3, label=name)
g = e2.groupby("msgs_per_alias")["chance_top1"].mean()
ax.plot(g.index, g.values, ls="--", color=MUTED, lw=1.2, label="chance")
ax.set_xscale("log", base=2); ax.set_xticks(ALIAS_SIZES); ax.set_xticklabels(ALIAS_SIZES)
ax.set_ylim(0, 1.05); ax.set_xlabel("Messages per alias"); ax.set_ylabel("Top-1 accuracy")
ax.set_title("Re-linking accuracy vs. messages per alias", loc="left")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=9)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig2_data_curve.png", dpi=200); plt.show()

In [ ]:
# ---- Experiment 3 (faster): disguise test, features computed once per seed × disguise ----
import time
def disguise_eval_multi(aliases, block_sets, disguise, seed=0):
    rng = np.random.default_rng(seed)
    al = aliases.copy().reset_index(drop=True)
    is_q = np.zeros(len(al), bool)
    for a, g in al.groupby("agent"):
        ix = g.index.to_numpy().copy(); rng.shuffle(ix)
        is_q[ix[: len(ix) // 2]] = True
    f = DISGUISES[disguise]
    al.loc[is_q, "texts"] = al.loc[is_q, "texts"].apply(lambda ts: [f(t) for t in ts])
    blocks = feature_blocks(al)
    q, k = np.where(is_q)[0], np.where(~is_q)[0]
    labels = al["agent"].to_numpy()
    counts = Counter(labels[k])
    chance = np.mean([counts[labels[i]] / len(k) for i in q])
    out = {}
    for name, bl in block_sets.items():
        S = combine(blocks, bl)
        nn = k[np.argmax(S[np.ix_(q, k)], axis=1)]
        out[name] = (np.mean(labels[nn] == labels[q]), chance)
    return out

E3_SETS = {n: ABLATIONS[n] for n in ["all features", "style only (function words + stats)"]}
rows, t0 = [], time.time()
for seed in SEEDS:
    al = CACHE[seed][0]
    for d in DISGUISES:
        for name, (acc, chance) in disguise_eval_multi(al, E3_SETS, d, seed).items():
            rows.append({"disguise": d, "features": name, "seed": seed, "accuracy": acc, "chance": chance})
        print(f"seed {seed}, {d}: done ({time.time()-t0:.0f}s so far)")
e3 = pd.DataFrame(rows); e3.to_csv(f"{RESULTS_DIR}/e3_disguise.csv", index=False)
E3 = e3.groupby(["features", "disguise"], sort=False)[["accuracy", "chance"]].agg(["mean", "std"]).round(3)
display(E3)

fig, ax = plt.subplots(figsize=(9.5, 3.8))
ds = list(DISGUISES); x = np.arange(len(ds)); w = 0.38
for i, name in enumerate(["all features", "style only (function words + stats)"]):
    g = e3[e3.features == name].groupby("disguise", sort=False)["accuracy"].agg(["mean", "std"]).reindex(ds)
    ax.bar(x + (i - 0.5) * w, g["mean"], w - 0.03, yerr=g["std"], color=PAL[i], label=name, error_kw={"ecolor": MUTED, "lw": 1})
ch = e3["chance"].mean(); ax.axhline(ch, color=MUTED, ls="--", lw=1.2); ax.text(len(ds) - 0.5, ch, " chance", color=MUTED, fontsize=9, va="bottom", ha="right")
ax.set_xticks(x); ax.set_xticklabels(ds, fontsize=9); ax.set_ylim(0, 1.05); ax.set_ylabel("Top-1 accuracy (disguised queries)")
ax.set_title("Re-linking accuracy when queries are disguised", loc="left")
ax.legend(fontsize=9, loc="upper left", bbox_to_anchor=(1.01, 1)); ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig3_disguise.png", dpi=200); plt.show()

In [ ]:
# ---- Experiment 4: telling apart agents built on the SAME model (closest to the incident) ----
fam_counts = msgs.groupby("family")["agent"].nunique()
print("Agents per model family:"); print(fam_counts)
rows = []
for seed in SEEDS:
    al, blocks, fams = CACHE[seed]
    S = combine(blocks, BLOCKS_ALL)
    obs, exp = wrong_neighbour_same_family(S, al["agent"], fams)
    r = evaluate(S, al["agent"], fams, estimate_k=False)
    rows.append({"seed": seed, "top1_all_candidates": r["top1"], "top1_same_family_only": r["same_family_top1"],
                 "chance_same_family": r["same_family_chance"], "queries_with_same_family_rivals": r["same_family_n"],
                 "mistakes_pointing_to_same_family": obs, "expected_if_random": exp})
e4 = pd.DataFrame(rows); e4.to_csv(f"{RESULTS_DIR}/e4_same_model.csv", index=False)
display(e4.drop(columns="seed").mean().round(3).to_frame("mean"))
if e4["mistakes_pointing_to_same_family"].isna().all():
    print("(No top-1 mistakes at all, so the 'mistakes' rows are empty.)")
if fam_counts.max() < 2:
    print("⚠️ No model family has 2+ agents here, so this test isn't meaningful for this village.")

In [ ]:
# ---- Experiment 5: does the way aliases are cut matter? (contiguous vs shuffled) ----
rows = []
for mode in ["contiguous", "random"]:
    for seed in SEEDS:
        al = make_aliases(msgs, MSGS_PER_ALIAS, mode, MAX_ALIASES_PER_AGENT, seed)
        blocks = feature_blocks(al)
        for name in ["all features", "style only (function words + stats)"]:
            r = evaluate(combine(blocks, ABLATIONS[name]), al["agent"], estimate_k=False)
            rows.append({"split": mode, "features": name, "seed": seed, **r})
e5 = pd.DataFrame(rows); e5.to_csv(f"{RESULTS_DIR}/e5_split_mode.csv", index=False)
display(e5.groupby(["features", "split"])[["top1", "pair_auc"]].agg(["mean", "std"]).round(3))
print("If 'random' is much higher than 'contiguous', part of the signal is topic drift over time, not identity.")

In [ ]:
# ---- Numbers for the report ----
def ms(df, col, **f):
    s = df
    for k, v in f.items(): s = s[s[k] == v]
    return f"{s[col].mean():.2f} ± {s[col].std():.2f}"
n_ag, n_al = e1["agents"].iloc[0], e1["aliases"].iloc[0]
lines = [
    f"DATA: {VILLAGE}, {n_ag} agents, {len(msgs)} messages after cleaning; {n_al} simulated aliases of {MSGS_PER_ALIAS} messages each; {len(SEEDS)} seeds.",
    f"E1 all features: top-1 {ms(e1,'top1',features='all features')} (chance {e1['chance_top1'].mean():.2f}), pair AUC {ms(e1,'pair_auc',features='all features')}, ARI {ms(e1,'ARI',features='all features')}.",
    f"E1 style only: top-1 {ms(e1,'top1',features='style only (function words + stats)')}.",
    f"E1 blind agent-count estimate: {e1[e1.features=='all features']['estimated_agents'].tolist()} vs true {n_ag}.",
    f"E2 curve (all features, top-1): " + ", ".join(f"{k} msgs → {v:.2f}" for k, v in e2[e2.features=='all features'].groupby('msgs_per_alias')['top1'].mean().items()),
    f"E3 disguise (all features): " + ", ".join(f"{k}: {v:.2f}" for k, v in e3[e3.features=='all features'].groupby('disguise', sort=False)['accuracy'].mean().items()),
    f"E3 disguise (style only): " + ", ".join(f"{k}: {v:.2f}" for k, v in e3[e3.features=='style only (function words + stats)'].groupby('disguise', sort=False)['accuracy'].mean().items()),
    f"E4 same-model rivals only: top-1 {e4['top1_same_family_only'].mean():.2f} (chance {e4['chance_same_family'].mean():.2f}); "
    f"{e4['mistakes_pointing_to_same_family'].mean():.0%} of mistakes point at a same-model agent (random would be {e4['expected_if_random'].mean():.0%}).",
    f"E5 split: contiguous {ms(e5,'top1',split='contiguous',features='all features')} vs random {ms(e5,'top1',split='random',features='all features')}.",
]
if USE_SYNTHETIC_DEMO:
    lines.insert(0, "⚠️ SYNTHETIC DEMO DATA. These numbers are NOT results. Set USE_SYNTHETIC_DEMO = False.")
report = "\n".join(lines); print(report)
open(f"{RESULTS_DIR}/key_numbers.txt", "w").write(report)
print("\nAll files saved in", RESULTS_DIR)

In [ ]:
# ---- Experiment 6: strict tests (reuses features from Experiment 1, ~1 minute) ----
# 6a. Time-gap test: a match only counts if it comes from a DIFFERENT time period than the query,
#     so the method can't win just by finding the next chunk of the same conversation.
# 6b. Exact-same-model test: only compare against agents running the IDENTICAL model string.
TEXT_BLOCKS = [b for b in BLOCKS_ALL if b != "timing"]
GAPS_DAYS = [0, 1, 7, 30]

id_col = find_col(agents_df, ["id", "agent_id"], required=False)
ms_col = find_col(agents_df, ["model_string", "model", "model_name"], required=False)
nm_col = find_col(agents_df, ["name", "display_name"], required=False)
exact_model, agent_name = {}, {}
for _, r in agents_df.iterrows():      # look agents up by id OR name, whichever the chat uses
    keys = [str(r[c]) for c in (id_col, nm_col) if c and pd.notna(r[c])]
    for k in keys:
        if ms_col and pd.notna(r[ms_col]): exact_model[k] = str(r[ms_col])
        if nm_col and pd.notna(r[nm_col]): agent_name[k] = str(r[nm_col])

rows_gap, rows_exact = [], []
for seed in SEEDS:
    al, blocks, fams = CACHE[seed]
    S = combine(blocks, TEXT_BLOCKS).copy(); np.fill_diagonal(S, -np.inf)
    labels = al["agent"].to_numpy()
    t = [to_time(pd.Series(ts)) for ts in al["times"]]
    start = np.array([x.min().value for x in t], dtype=float) / 8.64e13   # days
    end = np.array([x.max().value for x in t], dtype=float) / 8.64e13
    dist = np.maximum(0, np.maximum(start[None, :] - end[:, None], start[:, None] - end[None, :]))
    for gap in GAPS_DAYS:
        hits, base = [], []
        for i in range(len(labels)):
            pool = np.where((dist[i] > gap) & (np.arange(len(labels)) != i))[0]
            if not np.any(labels[pool] == labels[i]) or len(set(labels[pool])) < 2:
                continue
            j = pool[np.argmax(S[i, pool])]
            hits.append(labels[j] == labels[i]); base.append(np.mean(labels[pool] == labels[i]))
        rows_gap.append({"seed": seed, "min_gap_days": gap, "top1": np.mean(hits), "chance": np.mean(base),
                         "queries": len(hits), "share_of_aliases": len(hits) / len(labels)})
    if exact_model:
        em = np.array([exact_model.get(a, "?") for a in labels])
        hits, base = [], []
        for i in range(len(labels)):
            if em[i] == "?":
                continue
            pool = np.where((em == em[i]) & (np.arange(len(labels)) != i))[0]
            if len(set(labels[pool])) < 2 or not np.any(labels[pool] == labels[i]):
                continue
            j = pool[np.argmax(S[i, pool])]
            hits.append(labels[j] == labels[i]); base.append(np.mean(labels[pool] == labels[i]))
        rows_exact.append({"seed": seed, "top1_exact_same_model": np.mean(hits) if hits else np.nan,
                           "chance": np.mean(base) if base else np.nan, "queries": len(hits)})

e6a = pd.DataFrame(rows_gap); e6a.to_csv(f"{RESULTS_DIR}/e6a_time_gap.csv", index=False)
print("6a. Time-gap test (all text features):")
display(e6a.groupby("min_gap_days")[["top1", "chance", "queries", "share_of_aliases"]].mean().round(3))

print("\n6b. Exact-same-model test:")
if exact_model:
    inc = pd.Series({a: exact_model.get(a, "?") for a in msgs["agent"].unique()})
    shared = inc[inc != "?"].value_counts(); shared = shared[shared >= 2]
    print("Model strings shared by 2+ included agents:"); print(shared if len(shared) else "(none)")
    e6b = pd.DataFrame(rows_exact); e6b.to_csv(f"{RESULTS_DIR}/e6b_exact_model.csv", index=False)
    display(e6b.drop(columns="seed").mean().round(3).to_frame("mean"))

# Agent roster for the report appendix (AI model names only; no humans)
roster = summary[["messages", "family"]].copy()
roster.insert(0, "model_string", [exact_model.get(a, "?") for a in roster.index])
roster.insert(0, "name", [agent_name.get(a, "?") for a in roster.index])
roster.to_csv(f"{RESULTS_DIR}/agent_roster.csv"); display(roster.head(60))

fig, ax = plt.subplots(figsize=(7, 3.6))
g = e6a.groupby("min_gap_days")[["top1", "chance"]].agg(["mean", "std"])
xs = np.arange(len(GAPS_DAYS))
ax.errorbar(xs, g[("top1", "mean")], yerr=g[("top1", "std")], marker="o", ms=8, color=PAL[0], capsize=3, label="all text features")
ax.plot(xs, g[("chance", "mean")], ls="--", color=MUTED, lw=1.2, label="chance")
ax.set_xticks(xs); ax.set_xticklabels([f">{d} d" if d else "any" for d in GAPS_DAYS])
ax.set_ylim(0, 1.05); ax.set_xlabel("Minimum time gap between query and match"); ax.set_ylabel("Top-1 accuracy")
ax.set_title("Re-linking accuracy when matches must come from another time period", loc="left")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=9)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig4_time_gap.png", dpi=200); plt.show()

print("\nE6a: " + ", ".join(f"gap>{r.Index}d: {r.top1:.2f} (chance {r.chance:.2f})"
                            for r in e6a.groupby("min_gap_days")[["top1", "chance"]].mean().dropna().itertuples()))
if exact_model and len(rows_exact):
    print(f"E6b exact same model: top-1 {e6b['top1_exact_same_model'].mean():.2f} (chance {e6b['chance'].mean():.2f}, "
          f"{int(e6b['queries'].mean())} queries)")

In [ ]:
# ---- Experiment 7: deeper analysis (reuses Experiment 1 features; about 5-10 minutes) ----
# 7a  Hard mode: rivals only from the SAME era (agents join/leave over 19 months), and no adjacent chunks
# 7b  Leak audit: most agent-specific words, assigned-emoji use, and a re-run with emoji removed
# 7c  Model-family attribution for an agent the method has NEVER seen (which lab's model wrote this?)
# 7d  Investigator operating points: how many links can you trust at 90/95/99% precision?
# 7e  Counting agents blind, with the best features
# 7f  Which agents are easiest / hardest to re-identify, and why
# 7g  Fingerprint drift: does an agent's style change over months?
import re
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import squareform
from scipy.sparse.csgraph import connected_components
from scipy.sparse import csr_matrix
from scipy.stats import spearmanr
from sklearn.feature_extraction.text import CountVectorizer

TEXT_BLOCKS = [b for b in BLOCKS_ALL if b != "timing"]
FEATS = {"all text (no timing)": TEXT_BLOCKS, "style only": BLOCKS_STYLE_ONLY}
ERA_DAYS = 30
more = []   # lines for key_numbers_more.txt

# lookups by agent id OR name, whichever the chat uses
_id = find_col(agents_df, ["id", "agent_id"], required=False)
_nm = find_col(agents_df, ["name", "display_name"], required=False)
_em = "emoji" if "emoji" in agents_df.columns else None
NAME, EMOJI = {}, {}
for _, r in agents_df.iterrows():
    for k in [str(r[c]) for c in (_id, _nm) if c and pd.notna(r[c])]:
        if _nm and pd.notna(r[_nm]): NAME[k] = str(r[_nm])
        if _em and pd.notna(r[_em]) and str(r[_em]).strip(): EMOJI[k] = str(r[_em]).strip()
name_of = lambda a: NAME.get(a, a[:8])

def alias_days(al):
    t = [to_time(pd.Series(ts)) for ts in al["times"]]
    s = np.array([x.min().value for x in t], float) / 8.64e13
    e = np.array([x.max().value for x in t], float) / 8.64e13
    return s, e

def interval_gap(s, e):
    return np.maximum(0, np.maximum(s[None, :] - e[:, None], s[:, None] - e[None, :]))

def sim(blocks, bl):
    S = combine(blocks, bl).copy(); np.fill_diagonal(S, -np.inf); return S

# ===================== 7a. Hard mode =====================
rows = []
for seed in SEEDS:
    al, blocks, fams = CACHE[seed]
    lab = al["agent"].to_numpy(); n = len(lab)
    s, e = alias_days(al); mid = (s + e) / 2; G = interval_gap(s, e)
    near = np.abs(mid[:, None] - mid[None, :]) <= ERA_DAYS
    for fname, bl in FEATS.items():
        S = sim(blocks, bl)
        for min_gap in [1, 7]:
            hits, base, rivals = [], [], []
            for i in range(n):
                same = lab == lab[i]
                ok = near[i] & (~same | (G[i] > min_gap)); ok[i] = False
                pool = np.where(ok)[0]
                if not same[pool].any() or len(set(lab[pool])) < 2:
                    continue
                j = pool[np.argmax(S[i, pool])]
                hits.append(lab[j] == lab[i]); base.append(same[pool].mean()); rivals.append(len(set(lab[pool])) - 1)
            rows.append({"seed": seed, "features": fname, "min_gap_days": min_gap, "top1": np.mean(hits),
                         "chance": np.mean(base), "rival_agents_median": np.median(rivals), "coverage": len(hits) / n})
e7a = pd.DataFrame(rows); e7a.to_csv(f"{RESULTS_DIR}/e7a_hard_mode.csv", index=False)
print(f"7a. HARD MODE: rivals only from within ±{ERA_DAYS} days of the query; same-agent matches must be >N days away")
T = e7a.groupby(["features", "min_gap_days"])[["top1", "chance", "rival_agents_median", "coverage"]].mean().round(3)
display(T)
for (f, g), r in T.iterrows():
    more.append(f"7a hard mode [{f}, gap>{g}d]: top-1 {r.top1:.2f} vs chance {r.chance:.2f} "
                f"({r.top1 / r.chance:.1f}x), median {r.rival_agents_median:.0f} rival agents, coverage {r.coverage:.0%}")

# ===================== 7b. Leak audit =====================
print("\n7b. LEAK AUDIT: most agent-specific words (after name masking). Look for handles, emails, signatures.")
samp = msgs.sample(frac=1, random_state=0).groupby("agent").head(2000)   # up to 2,000 messages per agent
docs = samp.groupby("agent")["text"].apply(lambda x: "\n".join(x))
cv = CountVectorizer(token_pattern=r"(?u)\b\w[\w'.@-]*\w\b", max_features=80000)
X = cv.fit_transform(docs.values).toarray().astype(float); vocab = np.array(cv.get_feature_names_out())
tot = X.sum(0); nA = X.sum(1, keepdims=True); a0 = 0.5
za = np.log((X + a0) / (nA - X + a0)) - np.log((tot - X + a0) / (nA.sum() - nA - (tot - X) + a0))
za = za / np.sqrt(1 / (X + a0) + 1 / (tot - X + a0))
za[X < 20] = -np.inf
leak_rows = []
for k, a in enumerate(docs.index):
    top = vocab[np.argsort(-za[k])[:10]]
    leak_rows.append({"agent": name_of(a), "messages": int((msgs.agent == a).sum()), "most_distinctive_words": ", ".join(top)})
leak = pd.DataFrame(leak_rows).sort_values("messages", ascending=False)
leak.to_csv(f"{RESULTS_DIR}/e7b_distinctive_words.csv", index=False)
with pd.option_context("display.max_colwidth", 200):
    display(leak.head(25))

if EMOJI:
    er = []
    for a in msgs["agent"].unique():
        em = EMOJI.get(a)
        if not em:
            continue
        has = msgs["text"].str.contains(re.escape(em), regex=True)
        own, other = has[msgs.agent == a].mean(), has[msgs.agent != a].mean()
        er.append({"agent": name_of(a), "assigned_emoji": em, "own_use": own, "others_use": other,
                   "ratio": own / other if other > 0 else np.inf})
    er = pd.DataFrame(er).sort_values("own_use", ascending=False)
    er.to_csv(f"{RESULTS_DIR}/e7b_assigned_emoji.csv", index=False)
    print("\nAssigned avatar emoji: how often each agent uses its own vs. others use it")
    display(er.head(15).round(3))
    flagged = er[(er.own_use > 0.05) & (er.ratio > 5)]
    print(f"{len(flagged)} agents use their own assigned emoji as a possible signature (>5% of messages, >5x others).")

ASSIGNED = sorted(set(EMOJI.values()), key=len, reverse=True)
ASSIGNED_RE = re.compile("|".join(re.escape(x) for x in ASSIGNED)) if ASSIGNED else None
ALL_EMOJI_RE = re.compile("[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F000-\U0001F2FF️‍⭐⬆↔-↪⤴⤵〰〽㊗㊙]")
al0, blocks0, fams0 = CACHE[SEEDS[0]]
variants = {"as run": None}
if ASSIGNED_RE: variants["assigned emoji removed"] = lambda t: ASSIGNED_RE.sub("", t)
variants["all emoji removed"] = lambda t: ALL_EMOJI_RE.sub("", t)
rows = []
for vname, f in variants.items():
    if f is None:
        b = blocks0
    else:
        al_v = al0.copy(); al_v["texts"] = al_v["texts"].apply(lambda ts: [f(t) for t in ts]); b = feature_blocks(al_v)
    for fname, bl in FEATS.items():
        r = evaluate(combine(b, bl), al0["agent"], estimate_k=False)
        rows.append({"variant": vname, "features": fname, "top1": r["top1"], "pair_auc": r["pair_auc"]})
e7b = pd.DataFrame(rows); e7b.to_csv(f"{RESULTS_DIR}/e7b_emoji_masked.csv", index=False)
print("\nDoes accuracy survive removing emoji? (seed 0)"); display(e7b.round(3))
for _, r in e7b.iterrows():
    more.append(f"7b {r.variant} [{r.features}]: top-1 {r.top1:.2f}")

# ===================== 7c. Model-family attribution for unseen agents =====================
rows, fam_rows = [], []
for seed in SEEDS:
    al, blocks, fams = CACHE[seed]
    lab = al["agent"].to_numpy(); s, e = alias_days(al); mid = (s + e) / 2
    near = np.abs(mid[:, None] - mid[None, :]) <= ERA_DAYS
    S = sim(blocks, TEXT_BLOCKS)
    for era_matched in [False, True]:
        hits, base = [], []
        for i in range(len(lab)):
            if fams[i] == "unknown":
                continue
            ok = (lab != lab[i]) & (fams != "unknown")          # the agent itself is never in the pool
            if era_matched: ok &= near[i]
            pool = np.where(ok)[0]
            if len(pool) == 0 or not (fams[pool] == fams[i]).any():
                continue
            j = pool[np.argmax(S[i, pool])]
            hits.append(fams[j] == fams[i]); base.append((fams[pool] == fams[i]).mean())
            if seed == SEEDS[0] and era_matched: fam_rows.append({"family": fams[i], "hit": fams[j] == fams[i], "chance": base[-1]})
        rows.append({"seed": seed, "era_matched": era_matched, "family_acc": np.mean(hits), "chance": np.mean(base), "queries": len(hits)})
e7c = pd.DataFrame(rows); e7c.to_csv(f"{RESULTS_DIR}/e7c_family_attribution.csv", index=False)
print("\n7c. WHICH LAB'S MODEL WROTE THIS? Agent never seen before (all its aliases removed from the pool)")
display(e7c.groupby("era_matched")[["family_acc", "chance", "queries"]].mean().round(3))
fam_t = pd.DataFrame(fam_rows).groupby("family").agg(accuracy=("hit", "mean"), chance=("chance", "mean"), aliases=("hit", "size")).round(3)
print("By family (era-matched, seed 0):"); display(fam_t)
for em_, g in e7c.groupby("era_matched"):
    more.append(f"7c unseen-agent family attribution ({'era-matched' if em_ else 'all eras'}): "
                f"{g.family_acc.mean():.2f} vs chance {g.chance.mean():.2f}")

# ===================== 7d. Investigator operating points =====================
rows = []
for seed in SEEDS:
    al, blocks, fams = CACHE[seed]
    lab = al["agent"].to_numpy(); S = combine(blocks, TEXT_BLOCKS)
    iu = np.triu_indices(len(lab), 1); y = (lab[:, None] == lab[None, :])[iu]; sc = S[iu]
    o = np.argsort(-sc); y = y[o]; tp = np.cumsum(y); prec = tp / np.arange(1, len(y) + 1)
    r = {"seed": seed, "true_pairs": int(y.sum()), "all_pairs": len(y)}
    for k in [100, 1000, 10000]:
        r[f"precision@{k}"] = prec[k - 1] if k <= len(prec) else np.nan
    for target in [0.99, 0.95, 0.90]:
        ok = np.where(prec >= target)[0]
        r[f"recall@{int(target * 100)}%prec"] = tp[ok[-1]] / y.sum() if len(ok) else 0.0
    rows.append(r)
e7d = pd.DataFrame(rows); e7d.to_csv(f"{RESULTS_DIR}/e7d_operating_points.csv", index=False)
print("\n7d. INVESTIGATOR VIEW: rank all alias pairs by similarity and accept the top ones")
display(e7d.drop(columns="seed").mean().round(3).to_frame("mean"))
m = e7d.mean(numeric_only=True)
more.append(f"7d: of {m['all_pairs']:.0f} alias pairs ({m['true_pairs']:.0f} truly same-agent), top-1000 most similar are "
            f"{m['precision@1000']:.0%} correct; at 95% precision an investigator links {m['recall@95%prec']:.0%} of true pairs, "
            f"at 90% precision {m['recall@90%prec']:.0%}")

# ===================== 7e. Counting agents blind =====================
rows = []
for seed in SEEDS:
    al, blocks, fams = CACHE[seed]
    lab = al["agent"].to_numpy(); k_true = len(set(lab))
    for fname, bl in FEATS.items():
        S = combine(blocks, bl); D = np.clip(1 - S, 0, None); np.fill_diagonal(D, 0); D = (D + D.T) / 2
        Z = sch.linkage(squareform(D, checks=False), "average")
        best_k, best_s = None, -1
        for k in range(2, 3 * k_true + 1):
            cl = sch.fcluster(Z, k, "maxclust")
            if len(set(cl)) < 2: continue
            sv = silhouette_score(D, cl, metric="precomputed")
            if sv > best_s: best_k, best_s = len(set(cl)), sv
        rows.append({"seed": seed, "features": fname, "aliases": len(lab), "true_agents": k_true, "estimated_agents": best_k,
                     "ARI_at_true_k": adjusted_rand_score(lab, sch.fcluster(Z, k_true, "maxclust"))})
e7e = pd.DataFrame(rows); e7e.to_csv(f"{RESULTS_DIR}/e7e_blind_count.csv", index=False)
print("\n7e. HOW MANY AGENTS ARE BEHIND THESE NAMES? (blind clustering)")
T = e7e.groupby("features")[["aliases", "true_agents", "estimated_agents", "ARI_at_true_k"]].mean().round(2); display(T)
for f, r in T.iterrows():
    head = f"7e [{f}]: {r.aliases:.0f} names -> estimated {r.estimated_agents:.0f} agents (true {r.true_agents:.0f}); "
    if r.estimated_agents >= r.true_agents:
        collapsed = (r.aliases - r.estimated_agents) / (r.aliases - r.true_agents)
        more.append(head + f"removes {collapsed:.0%} of the redundant names but over-counts {r.estimated_agents / r.true_agents:.1f}x")
    else:
        more.append(head + f"under-counts (merges some agents together), {r.estimated_agents / r.true_agents:.1f}x of the true number")

# ===================== 7f. Per-agent breakdown =====================
al, blocks, fams = CACHE[SEEDS[0]]
lab = al["agent"].to_numpy(); S = sim(blocks, TEXT_BLOCKS); nn = S.argmax(1)
pa = pd.DataFrame({"agent": lab, "hit": lab[nn] == lab, "family": fams}).groupby("agent").agg(
    top1=("hit", "mean"), aliases=("hit", "size"), family=("family", "first"))
pa["messages"] = msgs["agent"].value_counts().reindex(pa.index).values
pa.insert(0, "name", [name_of(a) for a in pa.index]); pa = pa.sort_values("top1")
pa.to_csv(f"{RESULTS_DIR}/e7f_per_agent.csv")
rho, pv = spearmanr(np.log(pa["messages"]), pa["top1"])
print(f"\n7f. PER AGENT (seed 0, all text). Spearman(log messages, accuracy) = {rho:.2f} (p={pv:.3f})")
print("Hardest to re-identify:"); display(pa.head(8).round(2))
print("Easiest:"); display(pa.tail(8).round(2))
print("By family:"); display(pa.groupby("family")["top1"].agg(["mean", "count"]).round(2))
more.append(f"7f: per-agent top-1 ranges {pa.top1.min():.2f}-{pa.top1.max():.2f}; Spearman with log(messages) = {rho:.2f} (p={pv:.3f})")

# ===================== 7g. Fingerprint drift =====================
bins = [0, 1, 7, 30, 90, 10000]; blabels = ["<1 d", "1-7 d", "7-30 d", "30-90 d", ">90 d"]
rows = []
for seed in SEEDS:
    al, blocks, fams = CACHE[seed]
    lab = al["agent"].to_numpy(); s, e = alias_days(al); G = interval_gap(s, e)
    iu = np.triu_indices(len(lab), 1); same = (lab[:, None] == lab[None, :])[iu]; g = G[iu]
    for fname, bl in FEATS.items():
        sc = combine(blocks, bl)[iu]
        for lo, hi, bn in zip(bins[:-1], bins[1:], blabels):
            msk = (g >= lo) & (g < hi)
            rows.append({"seed": seed, "features": fname, "gap": bn,
                         "same_agent": sc[msk & same].mean() if (msk & same).any() else np.nan,
                         "different_agents": sc[msk & ~same].mean() if (msk & ~same).any() else np.nan,
                         "same_pairs": int((msk & same).sum())})
e7g = pd.DataFrame(rows); e7g.to_csv(f"{RESULTS_DIR}/e7g_drift.csv", index=False)
T = e7g.groupby(["features", "gap"], sort=False)[["same_agent", "different_agents", "same_pairs"]].mean().round(3)
print("\n7g. FINGERPRINT DRIFT: mean similarity of alias pairs by time apart"); display(T)

fig, ax = plt.subplots(figsize=(9, 3.8))
xs = np.arange(len(blabels))
for i, (fname, ls) in enumerate([("all text (no timing)", "-"), ("style only", "--")]):
    g = e7g[e7g.features == fname].groupby("gap", sort=False)[["same_agent", "different_agents"]].mean().reindex(blabels)
    ax.plot(xs, g["same_agent"], ls=ls, marker="o", ms=8, color=PAL[0], label=f"same agent ({fname})")
    ax.plot(xs, g["different_agents"], ls=ls, marker="o", ms=8, color=PAL[1], label=f"different agents ({fname})")
ax.set_xticks(xs); ax.set_xticklabels(blabels); ax.set_xlabel("Time between the two aliases"); ax.set_ylabel("Mean cosine similarity")
ax.set_title("Same-agent similarity vs. time between aliases", loc="left")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=9)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig5_drift.png", dpi=200); plt.show()
gt = e7g[e7g.features == "all text (no timing)"].groupby("gap", sort=False)[["same_agent", "different_agents"]].mean().reindex(blabels).dropna()
if len(gt):
    more.append("7g drift (all text, same vs different agents): " +
                ", ".join(f"{k}: {r.same_agent:.2f} vs {r.different_agents:.2f}" for k, r in gt.iterrows()))

txt = "\n".join(more)
if USE_SYNTHETIC_DEMO:
    txt = "⚠️ SYNTHETIC DEMO DATA. These numbers are NOT results.\n" + txt
open(f"{RESULTS_DIR}/key_numbers_more.txt", "w").write(txt)
print("\n" + "=" * 70 + "\n" + txt)

In [ ]:
# ---- Experiment 8: the closest thing to "copies of one model" in this data (~1 minute) ----
# 8a. Same weights, different scaffolding: model strings that match once "claude-code::" is removed
# 8b. Sibling variants: same version, different suffix (e.g. gpt-5.6-sol / -terra / -luna)
# 8c. Why are prolific agents harder to re-identify? Check the time span their aliases cover
import re
from scipy.stats import spearmanr
TEXT_BLOCKS = [b for b in BLOCKS_ALL if b != "timing"]
_id = find_col(agents_df, ["id", "agent_id"], required=False)
_nm = find_col(agents_df, ["name", "display_name"], required=False)
_ms = find_col(agents_df, ["model_string", "model", "model_name"], required=False)
MS, NM = {}, {}
for _, r in agents_df.iterrows():
    for k in [str(r[c]) for c in (_id, _nm) if c and pd.notna(r[c])]:
        if _ms and pd.notna(r[_ms]): MS[k] = str(r[_ms]).lower()
        if _nm and pd.notna(r[_nm]): NM[k] = str(r[_nm])
included = list(msgs["agent"].unique())
nm = lambda a: NM.get(a, a[:8])

def groups_from(keymap):
    s = pd.Series(keymap); s = s[s != "?"]
    return {k: list(v.index) for k, v in s.groupby(s) if len(v) >= 2}

GROUPS = {
    "8a same weights, different scaffolding": groups_from({a: MS.get(a, "?").replace("claude-code::", "") for a in included}),
    "8b sibling variants": groups_from({a: re.sub(r"-(sol|terra|luna|astra)$", "", MS.get(a, "?")) for a in included}),
}
GROUPS["8b sibling variants"] = {k: v for k, v in GROUPS["8b sibling variants"].items()
                                 if k not in GROUPS["8a same weights, different scaffolding"]}

def span_and_gap(al):
    t = [to_time(pd.Series(ts)) for ts in al["times"]]
    s = np.array([x.min().value for x in t], float) / 8.64e13
    e = np.array([x.max().value for x in t], float) / 8.64e13
    return s, e, np.maximum(0, np.maximum(s[None, :] - e[:, None], s[:, None] - e[None, :]))

rows, mrows = [], []
for seed in SEEDS:
    al, blocks, fams = CACHE[seed]
    lab = al["agent"].to_numpy(); S = combine(blocks, TEXT_BLOCKS).copy(); np.fill_diagonal(S, -np.inf)
    s, e, G = span_and_gap(al); nn = S.argmax(1)
    for test, groups in GROUPS.items():
        for gname, members in groups.items():
            idx = np.where(np.isin(lab, members))[0]
            for min_gap in [0, 1]:
                hits, base = [], []
                for i in idx:                       # rivals = ONLY the other members of the group
                    same = lab[idx] == lab[i]
                    pool = idx[(idx != i) & (~same | (G[i, idx] > min_gap))]
                    if not (lab[pool] == lab[i]).any() or len(set(lab[pool])) < 2:
                        continue
                    j = pool[np.argmax(S[i, pool])]
                    hits.append(lab[j] == lab[i]); base.append((lab[pool] == lab[i]).mean())
                if hits:
                    rows.append({"seed": seed, "test": test, "group": " / ".join(nm(a) for a in members),
                                 "min_gap_days": min_gap, "top1": np.mean(hits), "chance": np.mean(base), "queries": len(hits)})
            wrong = idx[lab[nn[idx]] != lab[idx]]     # in the FULL pool: do mistakes land on the twin/sibling?
            if len(wrong):
                exp = np.mean([(np.isin(lab, members) & (lab != lab[i])).sum() / (lab != lab[i]).sum() for i in wrong])
                mrows.append({"seed": seed, "test": test, "group": " / ".join(nm(a) for a in members),
                              "mistakes": len(wrong), "mistakes_to_group": np.isin(lab[nn[wrong]], members).mean(),
                              "expected_if_random": exp})

e8 = pd.DataFrame(rows); e8m = pd.DataFrame(mrows)
if len(e8):
    e8.to_csv(f"{RESULTS_DIR}/e8_same_model_groups.csv", index=False)
    print("Within-group re-linking (rivals are ONLY the other group members):")
    with pd.option_context("display.max_colwidth", 80):
        display(e8.groupby(["test", "group", "min_gap_days"])[["top1", "chance", "queries"]].mean().round(3))
else:
    print("No same-weights or sibling groups found among the included agents.")
if len(e8m):
    e8m.to_csv(f"{RESULTS_DIR}/e8_group_mistakes.csv", index=False)
    print("\nIn the full test, how often do this group's mistakes point at another group member?")
    with pd.option_context("display.max_colwidth", 80):
        display(e8m.groupby(["test", "group"])[["mistakes", "mistakes_to_group", "expected_if_random"]].mean().round(3))

# 8c. span check
al, blocks, fams = CACHE[SEEDS[0]]
lab = al["agent"].to_numpy(); S = combine(blocks, TEXT_BLOCKS).copy(); np.fill_diagonal(S, -np.inf)
s, e, _ = span_and_gap(al); nn = S.argmax(1)
pa = pd.DataFrame({"agent": lab, "hit": lab[nn] == lab, "start": s, "end": e}).groupby("agent").agg(
    top1=("hit", "mean"), first=("start", "min"), last=("end", "max"))
pa["span_days"] = pa["last"] - pa["first"]
pa["messages"] = msgs["agent"].value_counts().reindex(pa.index).values
r1 = spearmanr(np.log(pa.messages), pa.top1); r2 = spearmanr(pa.span_days, pa.top1); r3 = spearmanr(np.log(pa.messages), pa.span_days)
print(f"\n8c. Spearman: accuracy vs log(messages) {r1[0]:.2f} (p={r1[1]:.3f}); accuracy vs span of aliases {r2[0]:.2f} (p={r2[1]:.3f}); "
      f"log(messages) vs span {r3[0]:.2f}")
lines = []
if len(e8):
    for (t, g, mg), r in e8.groupby(["test", "group", "min_gap_days"])[["top1", "chance"]].mean().iterrows():
        lines.append(f"{t} [{g}, gap>{mg}d]: top-1 {r.top1:.2f} vs chance {r.chance:.2f}")
if len(e8m):
    for (t, g), r in e8m.groupby(["test", "group"])[["mistakes_to_group", "expected_if_random"]].mean().iterrows():
        lines.append(f"{t} [{g}]: {r.mistakes_to_group:.0%} of mistakes go to another group member (random {r.expected_if_random:.0%})")
lines.append(f"8c: accuracy vs log(messages) rho={r1[0]:.2f}; vs alias time span rho={r2[0]:.2f}; messages vs span rho={r3[0]:.2f}")
txt = "\n".join(lines)
if USE_SYNTHETIC_DEMO:
    txt = "⚠️ SYNTHETIC DEMO DATA. These numbers are NOT results.\n" + txt
open(f"{RESULTS_DIR}/key_numbers_e8.txt", "w").write(txt)
print("\n" + "=" * 70 + "\n" + txt)

## Outputs
Every experiment writes a CSV to `RESULTS_DIR`; the headline numbers are collected in `key_numbers.txt` (Experiments 1-5), `key_numbers_more.txt` (Experiments 6-7) and `key_numbers_e8.txt` (Experiment 8), and the dataset snapshot in `dataset_version.txt`. Clear all outputs before sharing this notebook: they contain excerpts of the AI Village data.